In [1]:
# %pip install --upgrade protobuf

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.preprocessing import StandardScaler

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from keras.layers import Dropout

from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from keras.optimizers import Adam

import random
import tensorflow as tf

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [3]:
SEED = 42
random.seed(SEED)           
np.random.seed(SEED)        
tf.random.set_seed(SEED) 

In [4]:
from pathlib import Path

CWD = Path.cwd()
PROJECT_ROOT = CWD.parent.parent
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"

print(PROCESSED_DIR)

C:\Users\Sameen\10 Pearls Project\AQI Predictor\Data\Processed


In [5]:
df = pd.read_csv(PROCESSED_DIR / "Aqi_Data_One_Day_Prediction_New.csv", index_col="time", parse_dates=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26232 entries, 2023-08-04 00:00:00+05:00 to 2026-07-31 23:00:00+05:00
Data columns (total 50 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   temperature_2m             26232 non-null  float64
 1   wind_speed_10m             26232 non-null  float64
 2   relative_humidity_2m       26232 non-null  int64  
 3   surface_pressure           26232 non-null  float64
 4   boundary_layer_height      21864 non-null  float64
 5   dew_point_2m               26232 non-null  float64
 6   precipitation              26232 non-null  float64
 7   cloud_cover                26232 non-null  int64  
 8   wind_direction_10m         26232 non-null  int64  
 9   wind_gusts_10m             26232 non-null  float64
 10  shortwave_radiation        26232 non-null  float64
 11  pm10                       26232 non-null  float64
 12  pm2_5                      26232 non-null  float64
 13 

# Dropping Null Columns

In [6]:
df = df.drop(columns = ["boundary_layer_height","f24_boundary_layer_height","wind_pollution_dispersion"])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26232 entries, 2023-08-04 00:00:00+05:00 to 2026-07-31 23:00:00+05:00
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   temperature_2m            26232 non-null  float64
 1   wind_speed_10m            26232 non-null  float64
 2   relative_humidity_2m      26232 non-null  int64  
 3   surface_pressure          26232 non-null  float64
 4   dew_point_2m              26232 non-null  float64
 5   precipitation             26232 non-null  float64
 6   cloud_cover               26232 non-null  int64  
 7   wind_direction_10m        26232 non-null  int64  
 8   wind_gusts_10m            26232 non-null  float64
 9   shortwave_radiation       26232 non-null  float64
 10  pm10                      26232 non-null  float64
 11  pm2_5                     26232 non-null  float64
 12  carbon_monoxide           26232 non-null  float64
 13  nitrogen_dioxi

In [8]:
# Seasonal cycle (saal ka)
df["cyclic_day_sin"] = np.sin(2 * np.pi * df.index.dayofyear / 365)
df["cyclic_day_cos"] = np.cos(2 * np.pi * df.index.dayofyear / 365)

# Weekly cycle (hafte ka)
df["cyclic_week_sin"] = np.sin(2 * np.pi * df.index.dayofweek / 7)
df["cyclic_week_cos"] = np.cos(2 * np.pi * df.index.dayofweek / 7)

# Daily cycle (din ka) — hourly data ke liye sabse zaroori
df["cyclic_hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["cyclic_hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Wind direction (0-360 degrees)
wd_rad = np.deg2rad(df["wind_direction_10m"])
df["cyclic_wind_sin"] = np.sin(wd_rad)
df["cyclic_wind_cos"] = np.cos(wd_rad)

# Purane raw columns hatao
df = df.drop(columns=["hour", "month", "dayofweek", "day", "quarter",
                      "wind_direction_10m"])

print(df.shape)   # (26232, 49) aana chahiye

(26232, 49)


# Splitting X and Y

In [9]:
X = df.drop(columns = ["Day_1_Future_AQI"])
Y = df["Day_1_Future_AQI"]

In [10]:
X.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 26232 entries, 2023-08-04 00:00:00+05:00 to 2026-07-31 23:00:00+05:00
Data columns (total 48 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   temperature_2m            26232 non-null  float64
 1   wind_speed_10m            26232 non-null  float64
 2   relative_humidity_2m      26232 non-null  int64  
 3   surface_pressure          26232 non-null  float64
 4   dew_point_2m              26232 non-null  float64
 5   precipitation             26232 non-null  float64
 6   cloud_cover               26232 non-null  int64  
 7   wind_gusts_10m            26232 non-null  float64
 8   shortwave_radiation       26232 non-null  float64
 9   pm10                      26232 non-null  float64
 10  pm2_5                     26232 non-null  float64
 11  carbon_monoxide           26232 non-null  float64
 12  nitrogen_dioxide          26232 non-null  float64
 13  sulphur_dioxid

In [11]:
Y.info()

<class 'pandas.core.series.Series'>
DatetimeIndex: 26232 entries, 2023-08-04 00:00:00+05:00 to 2026-07-31 23:00:00+05:00
Series name: Day_1_Future_AQI
Non-Null Count  Dtype  
--------------  -----  
26232 non-null  float64
dtypes: float64(1)
memory usage: 409.9 KB


In [12]:
Train = int(len(df)*0.8)

# Normalization Scaling

In [13]:
scX = StandardScaler()
scY = StandardScaler()

scX.fit(X.iloc[:Train])
scY.fit(Y.iloc[:Train].values.reshape(-1,1))

X_scaled = scX.transform(X)
Y_scaled = scY.transform(Y.values.reshape(-1,1))

In [14]:
print("X_scaled:", X_scaled.shape)   # (26232, 46)
print("Y_scaled:", Y_scaled.shape)   # (26232, 1)

X_scaled: (26232, 48)
Y_scaled: (26232, 1)


# Creating Sequence for Short Term Memory

In [15]:
BACK_LOOKY = 24
X_seq  = []
Y_seq = []

for i in range(BACK_LOOKY - 1, len(X)):
    X_seq .append(X_scaled[i-BACK_LOOKY + 1:i+1, :])
    Y_seq.append(Y_scaled[i])

X_seq  = np.array(X_seq)
Y_seq = np.array(Y_seq)

print(X_seq.shape)
print(Y_seq.shape)

(26209, 24, 48)
(26209, 1)


# Train and Testing Data

In [16]:
split = Train - BACK_LOOKY + 1

In [17]:
X_train = X_seq[:split]
X_test  = X_seq[split:]
Y_train = Y_seq[:split]
Y_test  = Y_seq[split:]

In [18]:
X_train.shape

(20962, 24, 48)

In [19]:
X_test.shape

(5247, 24, 48)

# LSTM Model Build

In [20]:
early_stop = EarlyStopping(monitor = 'val_loss',
                           patience = 20,
                           restore_best_weights = True)

checkpoint = ModelCheckpoint("best_epoch_lstm.keras",
                              monitor = 'val_loss',
                              save_best_only=True,
                              verbose = 0)

reduce = ReduceLROnPlateau(monitor = 'val_loss',
                           factor = 0.5,
                           patience = 8,
                           min_lr = 1e-6)

In [21]:
model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(0.3))

model.add(LSTM(32))
model.add(Dropout(0.3))

model.add(Dense(1))

model.compile(optimizer=Adam(learning_rate=0.0003), loss='mean_squared_error')
model.summary()

C:\Users\Sameen\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        28,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,377 (161.63 KB)

 Trainable params: 41,377 (161.63 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
final_model = model.fit(X_train,
                        Y_train, 
                        validation_split = 0.15, 
                        epochs = 200, 
                        batch_size = 64, 
                        shuffle = False, 
                        callbacks = [early_stop, reduce, checkpoint], 
                        verbose = 1)

Epoch 1/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 0.5984 - val_loss: 0.5871 - learning_rate: 3.0000e-04
Epoch 2/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - loss: 0.4119 - val_loss: 0.5269 - learning_rate: 3.0000e-04
Epoch 3/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - loss: 0.3502 - val_loss: 0.5085 - learning_rate: 3.0000e-04
Epoch 4/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - loss: 0.3137 - val_loss: 0.4809 - learning_rate: 3.0000e-04
Epoch 5/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - loss: 0.2790 - val_loss: 0.4889 - learning_rate: 3.0000e-04
Epoch 6/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 12s 28ms/step - loss: 0.2590 - val_loss: 0.4809 - learning_rate: 3.0000e-04
Epoch 7/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - loss: 0.2360 - val_loss: 0.4975 - learning_rate: 3.0000e-04
Epoch 8/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - loss: 0.2207 - val_loss: 0.4794 - learning_rate: 3.0000e-04
Epoch 9/200
279/279 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - loss: 0.2

# Evaluation Metrics|

In [23]:
prediction = model.predict(X_test)

predicted = scY.inverse_transform(prediction).ravel()
actual = scY.inverse_transform(Y_test).ravel()

r2   = r2_score(actual, predicted)
mae  = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))

print(f"R²   : {r2}")
print(f"MAE  : {mae}")
print(f"RMSE : {rmse}")

164/164 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
R²   : 0.5902577188193334
MAE  : 9.096418107012601
RMSE : 12.838032963655765
